# ETL TASK

### Step 1 – Remove Duplicates

In [0]:
df = spark.table("dbacademy.default.streaming_data")

In [0]:
print("Total Rows:", df.count())

print("Distinct Rows:", df.distinct().count())

Total Rows: 5000
Distinct Rows: 5000


### Step 2 – Handle Missing Genres

In [0]:
from pyspark.sql.functions import col

df.filter(col("genre").isNull()).show()

+---------+-------+-------+------------+---------------+-----+----------------+---------+-------+-------------------+
|stream_id|user_id|song_id|   song_name|         artist|genre|duration_seconds|  country| device|   stream_timestamp|
+---------+-------+-------+------------+---------------+-----+----------------+---------+-------+-------------------+
|        8|   1012|      3|    Senorita|     Ed Sheeran| NULL|             205|      USA| Mobile|2026-07-01 14:50:00|
|       16|   1462|      2|     Thunder|  Justin Bieber| NULL|             259|      USA|Desktop|2026-07-05 12:45:00|
|       17|   1181|      7|Shape of You|       Dua Lipa| NULL|             232|       UK| Tablet|2026-07-06 09:31:00|
|       20|   1197|      5|     Perfect|     Ed Sheeran| NULL|             173|      USA| Mobile|2026-07-04 14:21:00|
|       21|   1334|      3|    Believer|     The Weeknd| NULL|             286|       UK| Mobile|2026-07-06 20:33:00|
|       27|   1308|     10|  Bad Habits|Imagine Dragons|

In [0]:
#Now replace missing genres with "Unknown":
from pyspark.sql.functions import when

df = df.withColumn(
    "genre",
    when(col("genre").isNull(), "Unknown")
    .otherwise(col("genre"))
)

### Step 3 – Standardize Genre Names

In [0]:
from pyspark.sql.functions import initcap, lower

df = df.withColumn(
    "genre",
    initcap(lower(col("genre")))
)

### Step 4 – Verify the Changes

In [0]:
df.select("genre").distinct().show()

+-------+
|  genre|
+-------+
|    Pop|
|   Rock|
|Hip Hop|
|Unknown|
+-------+



### Step 5 – Save the Silver Layer

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_streams")